# T-condition behavioural screen (Qwen2.5-VL-3B)Plan: `docs/plan_v2.md` section 4, step 3 -- the cheapest kill-test. **Does anincongruent note move the model's answer on >= ~50 objects (D41)?**This notebook is a thin wrapper: all logic lives in `src/circuit_overlap/`.Run the cells in order. Cells marked **[FILL IN]** need information the repodoes not yet have, and **[VERIFY]** cells check an assumption before it cansilently corrupt the run.Settings: GPU T4 x2 (one is enough), Internet ON (needed to clone and to pull weights).

## 1. Clone the repo and install

In [ ]:
# [FILL IN] your repo URL (or upload the repo as a Kaggle dataset instead)REPO_URL = ""REPO_DIR = "/kaggle/working/research_paper"import os, subprocess, sysif REPO_URL and not os.path.exists(REPO_DIR):    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)sys.path.insert(0, f"{REPO_DIR}/src")print(subprocess.run(["git","-C",REPO_DIR,"rev-parse","--short","HEAD"],                     capture_output=True, text=True).stdout.strip())

In [ ]:
# Kaggle ships torch + transformers; Qwen2.5-VL needs a recent transformers.!pip install -q -U "transformers>=4.49" accelerateimport torch, transformersprint(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))

## 2. [FILL IN] Build the object manifest`configs/data.json` deliberately has `source_id: null` -- the Visual-Counterfactrelease is still unverified (plan 3.1, STATE.md). Write the adapter here, andrecord what you used back in `configs/data.json`.The manifest is a JSON list of`{object_id, image, original_color, counterfactual_color}`, using the**original-colour** image. Exclude `radish` and `spider` as Lietzow do.

In [ ]:
import json, pathlibMANIFEST = pathlib.Path("/kaggle/working/manifest.json")# [FILL IN] load Visual-Counterfact and write the manifest, e.g.:#   ds = load_dataset("<verified id>", split="train")#   items = [{"object_id": r["object"], "image": save_png(r["image_original"]),#             "original_color": r["color_original"],#             "counterfactual_color": r["color_counterfactual"]}#            for r in ds if r["object"] not in ("radish", "spider")]#   MANIFEST.write_text(json.dumps(items))from circuit_overlap.data import load_manifest, check_colors_in_vocabularyentries = load_manifest(MANIFEST)notes_cfg = json.loads(open(f"{REPO_DIR}/configs/notes.json").read())missing = check_colors_in_vocabulary(entries, notes_cfg["color_vocabulary"])print(len(entries), "objects;", "colours missing from vocabulary:", missing)assert not missing, "add these to configs/notes.json, else they can never be scored"

## 3. [VERIFY] Look at one attack image before spending GPU time

In [ ]:
from circuit_overlap.notes import NoteSpec, render_note, variant_texts, matched_neutral_wordfrom circuit_overlap.tokens import mask_to_token_indices, token_grid_shapefrom circuit_overlap.t_screen import load_notes_configcfg = load_notes_config(f"{REPO_DIR}/configs/notes.json")e = entries[0]img = e.open_image()texts = variant_texts(e.original_color, e.counterfactual_color,                      matched_neutral_word(e.counterfactual_color, cfg["neutral_words"]))for spec in cfg["specs"]:    r = render_note(img, texts["incongruent"], spec)    print(spec.name, "area", round(r.area_frac, 3), "font", r.font_name, r.font_size_px)    display(r.image.resize((256, 256)))

In [ ]:
# [VERIFY] the mask -> image-token mapping against the processor's real grid.# The model resizes the image, so map the mask at the SAME size the model sees.from transformers import AutoProcessorfrom circuit_overlap.t_screen import ScreenConfigconfig = ScreenConfig()           # [VERIFY] checkpoint id (STATE.md blocker 2)processor = AutoProcessor.from_pretrained(config.checkpoint,                                          min_pixels=config.pixels, max_pixels=config.pixels)enc = processor(text=["x"], images=[r.image], return_tensors="pt")print("grid_thw from processor:", enc["image_grid_thw"])print("our predicted token grid:", token_grid_shape(*r.image.size[::-1]))# These must agree (after the 2x2 merge) before any H3 attention claim.

## 4. [VERIFY] Colour words must be single, distinct first tokens (D26)

In [ ]:
from circuit_overlap.t_screen import single_first_token_idsids, bad = single_first_token_ids(processor.tokenizer, cfg["color_vocabulary"])print("ids:", ids)print("violating D26 (drop these from configs/notes.json):", bad)

## 5. Smoke run on 5 objects, then the full screen

In [ ]:
from circuit_overlap.t_screen import load_model, run_screenmodel, processor = load_model(config)records, summary = run_screen(entries[:5], model, processor, config, cfg)for r in records[:6]:    print(r.layout, r.object_id, "| neutral:", r.answer_neutral,          "| incongruent:", r.answer_incongruent,          "| d_shift:", round((r.delta_neutral or 0) - (r.delta_incongruent or 0), 2))

In [ ]:
import timet0 = time.time()records, summary = run_screen(entries, model, processor, config, cfg)print(f"{time.time()-t0:.0f}s for {len(records)} generations x2")summary

## 6. Save the result JSON (download it and commit it to `results/`)

In [ ]:
from dataclasses import asdictfrom circuit_overlap.provenance import result_envelope, write_resultenvelope = result_envelope(    experiment="t_behavioural_screen",    payload={"summary": summary, "records": [asdict(r) for r in records]},    model="Qwen2.5-VL-3B",    checkpoint=config.checkpoint,    seed=config.seed,    prompt=config.prompt_template,    n=summary["n_objects"],    repo_dir=REPO_DIR,    layouts=[s.name for s in cfg["specs"]],    dtype=config.dtype,    pixels=config.pixels,)path = write_result("/kaggle/working/t_screen_qwen25vl3b.json", envelope)print(path, summary["n_objects_usable"], "usable objects; gate_passed =", summary["gate_passed"])

## 7. What the outcome means (plan section 4)| Result | Next ||---|---|| `gate_passed = True` (>= 50 usable objects) | Proceed: T-circuit discovery || Too few flips, but `continuous_fallback.n_objects_shifted` is healthy | Use the continuous variant -- patch on logit-difference shifts without an argmax flip || Few flips and little shift | Raise note salience (size, contrast) via `configs/notes.json`, or try PaliGemma-3B as primary |Record the number in `docs/RESULTS.md` with model, n, seed, prompt and commit.Do not report it anywhere else first.